# Experimento: Predicciones de sentimiento con contexto de magnitud (Gemma)

**Objetivo**: comprobar si el sesgo detectado (el modelo evita la etiqueta "sideways" y fuerza
una dirección) se debe, al menos en parte, a que el titular por sí solo no permite distinguir un
movimiento de precio pequeño ("cede un 0,47%") de uno grande — el modelo solo ve un verbo
direccional, sin magnitud.

## ⚠️ Regla de integridad metodológica — LEE ESTO ANTES DE USAR LOS RESULTADOS

Este notebook añade al prompt la **magnitud** del movimiento de precio ese día (en valor
absoluto, ej. "0.47%"), pero **nunca el signo/dirección** — el modelo debe seguir infiriendo
por sí mismo si es bearish o bullish, solo a partir del texto. Aun así:

- **Las predicciones de este notebook NO deben usarse en las Etapas 3 y 5** (Métricas Financieras
  y Validación Temporal). Ambas miden si el sentimiento predicho se correlaciona con el retorno
  real — si el sentimiento se generó dándole al modelo información derivada de ese mismo retorno,
  la correlación saldría inflada de forma circular y esas métricas dejarían de significar nada.
- Este notebook es exclusivamente un **experimento de diagnóstico para la Etapa 2**: compara sus
  resultados contra `predicciones_sentimiento_gemma.json` (la versión sin magnitud) para ver si
  el sesgo hacia "sideways" mejora. Si mejora, es una pista de diseño de prompt útil para el
  futuro — no un reemplazo del archivo de predicciones "oficial".

### Documentos necesarios
1. `sentiment_ground_truth.csv` (titular, ticker, etc.)
2. `financial_ground_truth.csv` (para obtener `return_1d` y `enrich_status`)

In [ ]:
# @title 1. Instalación de dependencias
!pip install -q google-genai tqdm pandas

## 2. Subir archivos

In [ ]:
from google.colab import files
uploaded = files.upload()  # selecciona sentiment_ground_truth.csv Y financial_ground_truth.csv

## 3. API key desde los secretos de Colab

In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("✅ API key cargada desde los secretos de Colab.")
except userdata.SecretNotFoundError:
    raise RuntimeError("No encontré un secreto llamado 'GEMINI_API_KEY'. Añádelo con el icono 🔑.")
except userdata.NotebookAccessError:
    raise RuntimeError("El secreto existe pero no le has dado acceso a este notebook. Actívalo con el icono 🔑.")

import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

## 4. Configuración

In [ ]:
GROUND_TRUTH_PATH = "sentiment_ground_truth.csv"
FINANCIAL_PATH = "financial_ground_truth.csv"
OUTPUT_PATH = "predicciones_sentimiento_gemma_con_magnitud.json"          # nombre distinto a propósito
CHECKPOINT_PATH = "predicciones_sentimiento_gemma_con_magnitud_checkpoint.json"

MODEL = "gemma-4-31b-it"
MAX_OUTPUT_TOKENS = 1024

CHECKPOINT_EVERY = 25
MAX_RETRIES = 5
BASE_BACKOFF_SECONDS = 2

REQUESTS_PER_MINUTE_LIMIT = 10  # ajusta segun aistudio.google.com/rate-limit
MIN_SECONDS_BETWEEN_CALLS = 60 / REQUESTS_PER_MINUTE_LIMIT * 1.1

## 5. Cargar y combinar los dos ground truth (titular + magnitud de retorno)

Se calcula la magnitud en valor absoluto — **el signo se descarta explícitamente** antes de
construir el prompt, para no filtrar la dirección real.

In [ ]:
import pandas as pd

gt_sentiment = pd.read_csv(GROUND_TRUTH_PATH)
gt_financial = pd.read_csv(FINANCIAL_PATH)

df = gt_sentiment.merge(
    gt_financial[["id", "return_1d", "enrich_status"]],
    on="id", how="left",
)

# Magnitud SIN signo. Filas sin precio valido (enrich_status != 'ok') se quedan sin este dato.
df["magnitud_abs_1d"] = df["return_1d"].abs()
df.loc[df["enrich_status"] != "ok", "magnitud_abs_1d"] = None

n_con_magnitud = df["magnitud_abs_1d"].notna().sum()
print(f"Filas con contexto de magnitud disponible: {n_con_magnitud} de {len(df)}")
print(f"(el resto, {len(df) - n_con_magnitud}, usará el prompt sin magnitud, igual que la versión original)")

## 6. Cliente, prompt del sistema, y detección de compatibilidad con `thinking_config`

In [ ]:
from google import genai
from google.genai import types
from google.genai import errors

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

SYSTEM_PROMPT = """Eres un analista financiero senior, experto en interpretar el impacto de noticias en los mercados.
Tu tarea es clasificar el sentimiento de mercado de una noticia para un activo/ticker concreto.

Se te puede dar, como contexto adicional, la MAGNITUD (sin dirección) del movimiento de precio ese
día. Úsala solo para calibrar si el evento es significativo o rutinario — la dirección
(bullish/bearish) la debes inferir tú del propio texto de la noticia, nunca de la magnitud.

Responde ÚNICAMENTE con un objeto JSON, sin texto adicional antes ni después, con este formato exacto:
{"sentimiento": "bullish|bearish|sideways", "confianza": 0.0}

Reglas:
- "sentimiento" debe ser exactamente una de estas tres palabras: bullish, bearish, sideways.
- "confianza" es tu propia estimación (0.0 a 1.0) de cuán seguro estás de esa clasificación.
- No incluyas explicaciones, razonamiento ni texto fuera del JSON."""

BASE_CONFIG_KWARGS = dict(
    system_instruction=SYSTEM_PROMPT,
    temperature=0.0,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    response_mime_type="application/json",
)

def build_config(with_thinking_budget: bool, max_output_tokens: int = None):
    kwargs = dict(BASE_CONFIG_KWARGS)
    if max_output_tokens is not None:
        kwargs["max_output_tokens"] = max_output_tokens
    if with_thinking_budget:
        kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=0)
    return types.GenerateContentConfig(**kwargs)


print(f"Comprobando compatibilidad de '{MODEL}' con thinking_config...")
try:
    client.models.generate_content(model=MODEL, contents="Responde solo: ok", config=build_config(True))
    SUPPORTS_THINKING_CONFIG = True
    print("✅ El modelo admite thinking_config (desactivado con budget=0).")
except errors.ClientError as e:
    if "thinking" in e.message.lower():
        SUPPORTS_THINKING_CONFIG = False
        print("ℹ️  Este modelo no soporta thinking_config — se omite.")
    else:
        raise

GENERATION_CONFIG = build_config(SUPPORTS_THINKING_CONFIG)

## 7. Construcción del prompt (con magnitud opcional) y parseo

In [ ]:
import re
import json

def build_user_prompt(row):
    base = f"""Ticker: {row['ticker']}
Sector/Índice: {row['indice_sector']}
Fecha: {row['fecha']}
Titular de la noticia: "{row['titulo']}"""

    if pd.notna(row["magnitud_abs_1d"]):
        base += f"""

Contexto adicional: el precio del activo se movió un {row['magnitud_abs_1d']:.2f}% ese día
(sin indicar si fue al alza o a la baja — infiere la dirección solo del texto de la noticia)."""

    base += f"""

Clasifica el sentimiento de mercado de esta noticia para {row['ticker']}."""
    return base


def parse_response(raw_text):
    if raw_text is None:
        return None, None
    try:
        cleaned = raw_text.strip()
        cleaned = re.sub(r"^```(json)?|```$", "", cleaned, flags=re.MULTILINE).strip()
        data = json.loads(cleaned)
        sentimiento = str(data.get("sentimiento", "")).strip().lower()
        confianza = data.get("confianza", None)
        confianza = float(confianza) if confianza is not None else None
        return sentimiento, confianza
    except (json.JSONDecodeError, ValueError, AttributeError):
        pass

    lower = raw_text.lower()
    for kw in ["bullish", "alcista", "bearish", "bajista", "sideways", "lateral"]:
        if kw in lower:
            return kw, None
    return None, None

## 8. Prueba rápida — verifica que el prompt incluye la magnitud correctamente

In [ ]:
ejemplo_con_magnitud = df[df["magnitud_abs_1d"].notna()].iloc[0]
print("--- Ejemplo de prompt CON magnitud ---")
print(build_user_prompt(ejemplo_con_magnitud))

print("\n--- Prueba real contra el modelo ---")
test_response = client.models.generate_content(
    model=MODEL, contents=build_user_prompt(ejemplo_con_magnitud), config=GENERATION_CONFIG
)
print("Respuesta:", test_response.text)

## 9. Llamada con throttle, backoff, y distinción cuota diaria vs. rate-limit temporal

In [ ]:
import time
import random

class DailyQuotaExhausted(Exception):
    pass

RETRYABLE_CODES = {429, 500, 503}
_last_call_time = [0.0]


def extract_suggested_wait(message):
    match = re.search(r"retry in (\d+(\.\d+)?)s", message, flags=re.IGNORECASE)
    return float(match.group(1)) if match else None


def is_daily_quota_error(message):
    return "PerDay" in message or "GenerateRequestsPerDay" in message


def throttle():
    elapsed = time.time() - _last_call_time[0]
    if elapsed < MIN_SECONDS_BETWEEN_CALLS:
        time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)
    _last_call_time[0] = time.time()


def call_llm_with_backoff(user_prompt):
    token_budget = MAX_OUTPUT_TOKENS
    for attempt in range(MAX_RETRIES):
        throttle()
        config = GENERATION_CONFIG.model_copy(update={"max_output_tokens": token_budget})
        try:
            response = client.models.generate_content(model=MODEL, contents=user_prompt, config=config)
            if response.text is None:
                finish_reason = response.candidates[0].finish_reason if response.candidates else None
                if finish_reason == types.FinishReason.MAX_TOKENS:
                    token_budget *= 2
                    print(f"  Sin espacio para responder (MAX_TOKENS). Reintentando con {token_budget} tokens...")
                    continue
                raise RuntimeError(f"Respuesta vacía del modelo (finish_reason={finish_reason})")
            return response.text
        except (errors.ClientError, errors.ServerError) as e:
            if e.code not in RETRYABLE_CODES:
                raise RuntimeError(f"Error no recuperable ({e.code}): {e.message}") from e
            if e.code == 429 and is_daily_quota_error(e.message):
                raise DailyQuotaExhausted(e.message)
            suggested_wait = extract_suggested_wait(e.message)
            wait = suggested_wait if suggested_wait else BASE_BACKOFF_SECONDS * (2 ** attempt)
            wait += random.uniform(0, 1)
            etiqueta = "Límite de cuota" if e.code == 429 else "Servidor saturado (temporal, no es cuota)"
            print(f"  {etiqueta} ({e.code}). Esperando {wait:.1f}s...")
            time.sleep(wait)
    raise RuntimeError(f"Fallaron los {MAX_RETRIES} reintentos para esta fila.")

## 10. Checkpointing (solo cuenta como "hecha" una fila con predicción exitosa)

In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            data = json.load(f)
        exitosas = [r for r in data if r["sentimiento_predicho"] is not None]
        ids_procesados = {r["id"] for r in exitosas}
        print(f"Checkpoint encontrado: {len(exitosas)} filas exitosas, se reanuda desde ahí.")
        return exitosas, ids_procesados
    return [], set()


def save_checkpoint(resultados):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

## 11. Bucle principal

In [ ]:
from tqdm.auto import tqdm

resultados, ids_procesados = load_checkpoint()
pendientes = df[~df["id"].isin(ids_procesados)]
print(f"Total filas: {len(df)} | Ya procesadas con éxito: {len(ids_procesados)} | Pendientes: {len(pendientes)}")

detenido_por_cuota_diaria = False

for i, (_, row) in enumerate(tqdm(pendientes.iterrows(), total=len(pendientes), desc="Clasificando (con magnitud)")):
    user_prompt = build_user_prompt(row)
    try:
        raw_text = call_llm_with_backoff(user_prompt)
        sentimiento, confianza = parse_response(raw_text)
        resultados.append({
            "id": int(row["id"]), "sentimiento_predicho": sentimiento,
            "confianza": confianza, "respuesta_cruda": raw_text,
            "tuvo_contexto_magnitud": bool(pd.notna(row["magnitud_abs_1d"])),
        })
    except DailyQuotaExhausted:
        save_checkpoint(resultados)
        print(f"\n🛑 Cuota DIARIA agotada en la fila id={row['id']}. Progreso guardado: {len(resultados)} filas.")
        detenido_por_cuota_diaria = True
        break
    except RuntimeError as e:
        print(f"  ⚠️ Fila id={row['id']} falló tras reintentos: {e}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(resultados)

save_checkpoint(resultados)
print(f"\nProgreso actual: {len(resultados)} de {len(df)} filas procesadas con éxito.")
if detenido_por_cuota_diaria:
    print("(Proceso detenido por cuota diaria agotada, no completado.)")

## 12. Guardar y descargar

In [ ]:
ids_con_prediccion = {r["id"] for r in resultados}
resultados_completos = list(resultados)
for id_faltante in df.loc[~df["id"].isin(ids_con_prediccion), "id"]:
    resultados_completos.append({
        "id": int(id_faltante), "sentimiento_predicho": None,
        "confianza": None, "respuesta_cruda": None, "tuvo_contexto_magnitud": None,
    })
resultados_completos.sort(key=lambda r: r["id"])

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados_completos, f, ensure_ascii=False, indent=2)

n_exitosas = sum(1 for r in resultados_completos if r["sentimiento_predicho"] is not None)
print(f"Guardado {OUTPUT_PATH}: {n_exitosas} de {len(resultados_completos)} filas con predicción.")
print("\n⚠️ Recuerda: este archivo es solo para comparar contra la versión sin magnitud en la Etapa 2.")
print("   NO lo uses en las Etapas 3 y 5 (Métricas Financieras / Validación Temporal).")

from google.colab import files as colab_files
colab_files.download(OUTPUT_PATH)

## 13. Comparación rápida contra la versión sin magnitud

Ejecuta esto tras terminar, para ver de inmediato si el sesgo hacia "sideways" mejoró, antes de
pasar los resultados por el notebook completo de evaluación.

In [ ]:
from google.colab import files as colab_files

print("Sube predicciones_sentimiento_gemma.json (la version ORIGINAL, sin magnitud) para comparar:")
uploaded_original = colab_files.upload()

original = pd.DataFrame(json.load(open(list(uploaded_original.keys())[0], encoding="utf-8")))
nueva = pd.DataFrame(resultados_completos)

comparacion = gt_sentiment[["id", "regimen_mercado"]].merge(
    original[["id", "sentimiento_predicho"]].rename(columns={"sentimiento_predicho": "pred_original"}),
    on="id", how="inner"
).merge(
    nueva[["id", "sentimiento_predicho"]].rename(columns={"sentimiento_predicho": "pred_con_magnitud"}),
    on="id", how="inner"
)

for col, nombre in [("pred_original", "SIN magnitud (original)"), ("pred_con_magnitud", "CON magnitud (nuevo)")]:
    sub = comparacion[comparacion["regimen_mercado"] == "sideways"]
    acierto_sideways = (sub[col] == "sideways").mean()
    print(f"{nombre}: acierto en noticias realmente 'sideways' = {acierto_sideways:.1%}")
    print(f"  Distribución de lo predicho para esas noticias: {sub[col].value_counts(normalize=True).round(3).to_dict()}")
    print()